In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
def Profit_for_2MA(dma_low,dma_high,num_of_candle,df):
    slct=['Date','ClosePrice']
    max_close=(df
        .withColumn('Date',to_timestamp(col('Date'),'dd-MMM-yyyy'))
        .orderBy(col('Date').desc())
        .limit(1)
        .select('ClosePrice')
        .collect()[0][0]
        )
    # print(max_close)
    profit=(df
        .select(slct)
        .withColumn('ClosePrice',col('ClosePrice').cast('float'))
        # .withColumn('Date',to_timestamp(col('Date'),'dd-MMM-yyyy'))
        .withColumn('dma_5', avg('ClosePrice').over(Window.orderBy('Date').rowsBetween(1-dma_low, 0)))
        .withColumn('dma_10', avg('ClosePrice').over(Window.orderBy('Date').rowsBetween(1-dma_high, 0)))
        .orderBy(col('Date').desc())
        .limit(num_of_candle)
        .orderBy(col('Date').asc())
        .withColumn('dma_diff_5_10',col('dma_5')-col('dma_10'))
        .withColumn('changeflag',col('dma_diff_5_10')*lag('dma_diff_5_10',1).over(Window.orderBy(col('Date'))))
        .withColumn('changeflag',when(col('changeflag')>0,lit(None)).otherwise(col('changeflag')))
        .withColumn('check',col('dma_diff_5_10')*col('changeflag'))
        .filter(col('check').isNotNull())
        .withColumn('Stratergy',when(col('check')>0,lit('SELL')).otherwise(lit('BUY')))
        .select('Date','Stratergy','ClosePrice')
        .withColumn('lead',lead(col('ClosePrice'),1,max_close).over(Window.orderBy(col('Date').asc())))
        .withColumn('Profit',when(col('Stratergy')=='BUY',col('lead')-col('ClosePrice')).otherwise(lit(None)))
        .agg(sum(col('Profit')).alias('TotalProfit'),sum(when(col('Profit')<0,1).otherwise(0)).alias('No of Loses'),sum(when(col('Profit')>0,1).otherwise(0)).alias('No of Profits'))
        # .collect()[0][0]
    )
    


    enddate=df.withColumn('Date',to_timestamp(col('Date'),'dd-MMM-yyyy')).agg(max(col('Date'))).collect()[0][0]
    row=[enddate,num_of_candle,dma_low,dma_high,profit]
    # row=[int(num_of_candle),int(dma_low),int(dma_high),float(profit)]
    profit_df=(profit
        .withColumn('EndDate',lit(enddate))
        .withColumn('Num_of Candles',lit(num_of_candle))
        .withColumn('dma_low',lit(dma_low))
        .withColumn('dma_high',lit(dma_high))
        .select('EndDate','TotalProfit','dma_low','dma_high','No of Loses','No of Profits')
            )
    return profit_df


In [0]:
df=(spark.read.parquet('/Volumes/workspace/default/projecttradevolume/1_min/INFY/2026/01/**')
    .selectExpr(
        'DateTime AS Date',
        'close As ClosePrice'
    )
)

In [0]:
v_dma_low=100
v_dma_high=100
p_df=None
for low in range(5,v_dma_low+1,5):
    for high in range(10,v_dma_high+1,5): 
        if high > low:
            # print(low,high)
            if p_df is None:
                p_df=Profit_for_2MA(low,high,375,df) 
            else:
                p_df=p_df.union(Profit_for_2MA(low,high,375,df))
display(p_df)